In [2]:
import jax
from jax import random
from scipy.stats import *
from scipy import stats 
from typing import Union, List, Literal, TypeAlias
import numpy as np
import numpy.typing as npt
import pandas as pd
from functools import wraps, partial
from arch import arch_model
import time
from dataclasses import dataclass
from jax.scipy.optimize import minimize
import matplotlib.pyplot as plt
from math import exp, pi, sqrt

import jax.numpy as jnp

jax.config.update("jax_enable_x64", True) #for better numerical stability 

number = Union[int, float]
number_like = Union[List[number], number]
array_like = Union[List[number], np.ndarray]
distributions: TypeAlias = Literal["chauchy", "chi2", "expon", "exponpow", "gamma", "lognorm", "norm", "powerlaw", "rayleigh",
                            "uniform", "t", "gumbel_r", "f"]  
FloatArray = npt.NDArray[np.float64]
Floats = Union[float, FloatArray]
Int = Union[int, np.int16, np.int32, np.int64, jnp.int64, jnp.int32, jnp.int16]

from Vares import Auxiliary

In [ ]:
@dataclass
class GBMParams: 
    volatility: Floats 
    mean: Floats 

def make_gbm_silumator(
        params: GBMParams,
        T: int, 
        granularity: int = 1000
    ): 
    def simulate(n_paths: int, seed: int = 0xB0BA_C_3AB0DA): 
        time_grid = T * granularity
        dt = 1 / granularity
        random = np.random.default_rng(seed)
        norm = random.normal(size=(n_paths, time_grid))

        #Simulate log returns paths using GBM.
        d_log_S = (
            (params.mean - ((params.volatility**2) / 2)) * dt + 
            + params.volatility * norm * np.sqrt(dt)
        )
        d_log_S = np.insert(d_log_S, 0, np.zeros(n_paths), axis=1)
            
        #convert log returns to simple terminal returns (at date t = T): 
        terminal_returns = np.exp(np.cumsum(d_log_S, axis=-1)) - 1
        return terminal_returns, d_log_S 
    return simulate

class Portfolio():
    '''(IN WORK)'''
    def __init__(self, portfolio_returns, **kwargs): 
        """For now portfolio returns of size 1 x N is assumed"""
        self.portfolio_returns = portfolio_returns 
        try:
            self.number_of_assets = self.portfolio_returns.shape[1] 
        except IndexError: 
            self.number_of_assets = 1


        self.kwargs = kwargs.copy()

    def simulate(self, gbm: GBMParams, T: int =10): 
        '''Simple Monte-Carlo simulation of the future returns.''' 
        sim = make_gbm_silumator(gbm, T)
        self.sim = sim
        terminal_returns, paths = sim(1000)

        self.paths = paths # for testing 

        return terminal_returns

    def calibrate(self, horizon=1, **kwargs):
        '''The method implies Geometric Brownian Motion parameters (mean and volatility) from the available data using GARCH(p, q).
        
        !!! As now it uses arch package script can not handle portfolio altogether: needs portfolio_returns of size 1 x N . 
        Requires Update.'''
        am = arch_model(self.portfolio_returns, vol='Garch', dist='normal', **kwargs)
        result = am.fit(disp='off')
        forecast = result.forecast(horizon=horizon)
        mu = result.params.get("mu", 0)
        sigma = forecast.variance.values[-1, 0] ** 0.5
        parameters = GBMParams(volatility=sigma, mean=mu)
        return parameters
    
    def historical_var(self, alpha=0.01):
        params = Portfolio.calibrate(self)
        returns = Portfolio.simulate(self, params)

        return -np.quantile(returns, alpha)


In [49]:
normal_returns = stats.norm.rvs(size=1000)
_ = np.array(normal_returns)
print(_.shape)
portfolio = Portfolio(_)
print(portfolio.historical_var())

(1000,)
0.9999496312789632


In [47]:
from Vares import historical_var
print(historical_var(normal_returns))

historical_var() took 0.713999s
2.1602515981026706
